# SDE-Net diretto t+6h — evento 28–29 giugno 2019

Analisi post-hoc del modello già addestrato. Il notebook **non rilancia il training** e usa tutte le predizioni disponibili, senza campionamento.

La diagnostica mostra la risposta temporale del modello e confronta separatamente 28 giugno, 29 giugno e condizioni normali nelle cinque fasce di produzione.

In [ ]:
from pathlib import Path
import importlib
import sys

from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'physiq_pv').is_dir():
    raise FileNotFoundError('Avviare il notebook dalla root del repository o da notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import physiq_pv.reporting.posthoc_outputs as posthoc_outputs
import physiq_pv.experiments.sde_pipeline as pipe

importlib.reload(posthoc_outputs)
pipe = importlib.reload(pipe)

RUN_NAME = 'pvgis_stgnn_paper_faithful_gaussian_detector_mtgflow_ep60_h1-2-3-4-5-6_direct_seed1'
FORECAST_HORIZON = 6
out_dir = ROOT / 'outputs' / RUN_NAME
predictions_path = out_dir / 'predictions.csv'
posthoc_dir = out_dir / 'posthoc_by_horizon' / f't_plus_{FORECAST_HORIZON}'
reference_peak_path = posthoc_dir / 'reference_production_peaks.csv'

for required in (predictions_path, reference_peak_path):
    if not required.is_file():
        raise FileNotFoundError(required)

print('Run SDE-Net:', out_dir)
print('Predizioni:', predictions_path)

## Risposta temporale del modello

La diagnostica usa tutte le 1.149 località. Le aree rosse identificano le ore classificate come evento regionale raro dal protocollo stagionale P97.5.

In [ ]:
june_event = pipe.build_extreme_event_diagnostic(
    out_dir,
    start='2019-06-28',
    end='2019-06-30',  # estremo escluso: include tutto il 28 e 29 giugno
    figure_subdir='events/t_plus_6/june_extreme_event',
    horizon_hours=FORECAST_HORIZON,
)
print(f"Righe usate (nessun campionamento): {june_event['rows_used']:,}")
print('Soglia regionale stagionale:', june_event['regional_threshold'])
display(june_event['summary'])
display(june_event['hourly'])
display(Image(filename=str(june_event['figure_path'])))
print('CSV orario:', june_event['hourly_path'])
print('CSV riepilogo:', june_event['summary_path'])

## Normale 2019 vs 28 giugno vs 29 giugno

Boxplot, istogrammi e metriche usano tutte le predizioni diurne valide e sono separati nelle fasce 0–20%, 20–40%, 40–60%, 60–80% e 80–100% della potenza di riferimento.

In [ ]:
june_comparison = pipe.build_extreme_event_comparison_figures(
    out_dir,
    event_dates=('2019-06-28', '2019-06-29'),
    comparison_name='june_extreme_28_29_t_plus_6',
    figure_subdir='events/t_plus_6/june_extreme_event',
    horizon_hours=FORECAST_HORIZON,
    reference_peak_path=reference_peak_path,
)
display(june_comparison['metrics'])
print('CSV metriche:', june_comparison['metrics_path'])
print('Grafici creati:', len(june_comparison['figure_paths']))
for figure_path in june_comparison['figure_paths'].values():
    display(Image(filename=str(figure_path)))